# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.6 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64, pickle
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
torch.set_num_threads(1)

In [6]:
TASK_ID = 'task356'
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path('/mnt/data/task356(1).json')
KAGGLE_TASK_JSON = Path(COMPETITION) / f'{TASK_ID}.json'
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT_DIR = WORKDIR / f'{TASK_ID}_revised_onnx'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f'{TASK_ID}.onnx'
SUBMISSION_PATH = WORKDIR / 'submission.zip'
SUMMARY_PATH = OUT_DIR / f'{TASK_ID}_validation_summary.json'

with TASK_JSON.open() as f:
    task = json.load(f)

print('Task:', TASK_ID)
print('Task JSON:', TASK_JSON)
print('Splits:', {k: len(task.get(k, [])) for k in ['train', 'test', 'arc-gen']})

Task: task356
Task JSON: /kaggle/input/competitions/neurogolf-2026/task356.json
Splits: {'train': 3, 'test': 1, 'arc-gen': 262}


In [7]:
def grid_to_tensor_zero_padded(grid, h=H, w=W, ch=CH):
    arr=np.zeros((1,ch,h,w),dtype=np.float32)
    for r,row in enumerate(grid):
        for c,v in enumerate(row):
            arr[0,int(v),r,c]=1.0
    return arr


def compose_output(active, masks):
    # masks dict k -> [B,1,H,W], nonzero colors. Assumes no overlaps or deterministic priority by later overwrite impossible in tensors.
    outs=[]; occ=torch.zeros_like(active)
    for k in range(1,10):
        m=masks.get(k, torch.zeros_like(active))
        m=(m>0.5).float()*active
        outs.append(m)
        occ=torch.clamp(occ+m,0,1)
    ch0=active*(1.0-torch.clamp(occ,0,1))
    return torch.cat([ch0]+outs, dim=1)*active

RULE_DESCRIPTION = 'generic connect all same-colored seed points that share a row or column'


In [8]:
class Task356Generic(nn.Module):
    def forward(self,x):
        active=(x.sum(dim=1,keepdim=True)>0.5).float()
        masks={}
        for k in range(1,10):
            m=x[:,k:k+1]*active
            row_count=m.sum(dim=3,keepdim=True)
            left=(m.cumsum(dim=3)>0.5).float()
            right=(torch.flip(torch.flip(m,dims=[3]).cumsum(dim=3),dims=[3])>0.5).float()
            row_fill=left*right*(row_count>=1.5).float()
            col_count=m.sum(dim=2,keepdim=True)
            up=(m.cumsum(dim=2)>0.5).float()
            down=(torch.flip(torch.flip(m,dims=[2]).cumsum(dim=2),dims=[2])>0.5).float()
            col_fill=up*down*(col_count>=1.5).float()
            masks[k]=torch.clamp(m+row_fill+col_fill,0,1)*active
        return compose_output(active,masks)


model = Task356Generic().eval()
print(model)

Task356Generic()


In [9]:
dummy = torch.from_numpy(grid_to_tensor_zero_padded(task['test'][0]['input']))

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    input_names=['input'],
    output_names=['output'],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))

print('ONNX path:', ONNX_PATH)
print('ONNX size:', ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/2704130916.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX path: /kaggle/working/task356_revised_onnx/task356.onnx
ONNX size: 86524


In [10]:
def vi_shape(vi):
    dims = []
    for d in vi.type.tensor_type.shape.dim:
        if d.dim_value:
            dims.append(int(d.dim_value))
        elif d.dim_param:
            dims.append(str(d.dim_param))
        else:
            dims.append(None)
    return dims

onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
empty_inputs = [(node.name, node.op_type, list(node.input)) for node in onnx_model.graph.node if any(inp == '' for inp in node.input)]
bad_shapes = []
for vi in list(onnx_model.graph.input) + list(onnx_model.graph.value_info) + list(onnx_model.graph.output):
    shp = vi_shape(vi)
    if any(d is None or isinstance(d, str) for d in shp):
        bad_shapes.append((vi.name, shp))

print('input shape:', vi_shape(onnx_model.graph.input[0]))
print('output shape:', vi_shape(onnx_model.graph.output[0]))
print('ONNX size:', ONNX_PATH.stat().st_size)
print('ops:', dict(ops))
print('forbidden ops:', sorted(forbidden & set(ops)))
print('empty optional inputs:', len(empty_inputs))
print('non-static tensor shapes:', len(bad_shapes))

assert vi_shape(onnx_model.graph.input[0]) == [1, 10, 30, 30]
assert vi_shape(onnx_model.graph.output[0]) == [1, 10, 30, 30]
assert not (forbidden & set(ops))
assert not empty_inputs
assert ONNX_PATH.stat().st_size < 1_440_000

input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
ONNX size: 86524
ops: {'Constant': 323, 'ReduceSum': 19, 'Greater': 46, 'Cast': 64, 'Slice': 45, 'Mul': 65, 'CumSum': 36, 'GreaterOrEqual': 18, 'Add': 27, 'Clip': 19, 'Sub': 1, 'Concat': 1}
forbidden ops: []
empty optional inputs: 0
non-static tensor shapes: 0


In [11]:
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=['CPUExecutionProvider'])

def validate_examples(examples):
    tensor_ok = 0
    grid_ok = 0
    outside_zero_ok = 0
    bad = []
    for i, ex in enumerate(examples):
        x = grid_to_tensor_zero_padded(ex['input'])
        y = sess.run(None, {'input': x})[0]
        exp = grid_to_tensor_zero_padded(ex['output'])
        pred_bin = (y > 0.5).astype(np.float32)
        if np.array_equal(pred_bin, exp):
            tensor_ok += 1
        else:
            bad.append(i)
        h, w = len(ex['output']), len(ex['output'][0])
        pred_grid = pred_bin[0, :, :h, :w].argmax(axis=0).astype(np.int64).tolist()
        if pred_grid == ex['output']:
            grid_ok += 1
        input_active = x.sum(axis=1, keepdims=True) > 0.5
        if np.all(np.abs(y * (~input_active)) < 1e-5):
            outside_zero_ok += 1
    return {
        'tensor_exact_zero_padded': [tensor_ok, len(examples)],
        'grid_argmax_inside_canvas': [grid_ok, len(examples)],
        'outside_input_active_all_channels_zero': [outside_zero_ok, len(examples)],
        'bad_indices': bad[:10],
    }

def validate_split(split):
    return validate_examples(task[split])

rng = random.Random(0)
arcgen_indices = list(range(len(task.get('arc-gen', []))))
rng.shuffle(arcgen_indices)
holdout_n = max(1, int(math.ceil(0.60 * len(arcgen_indices)))) if arcgen_indices else 0
arcgen_holdout = [task['arc-gen'][i] for i in arcgen_indices[:holdout_n]]

summary = {
    'task_id': TASK_ID,
    'rule': RULE_DESCRIPTION,
    'onnx_path': str(ONNX_PATH),
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'input_shape': vi_shape(onnx_model.graph.input[0]),
    'output_shape': vi_shape(onnx_model.graph.output[0]),
    'ops': dict(ops),
    'forbidden_ops': sorted(forbidden & set(ops)),
    'empty_optional_inputs': len(empty_inputs),
    'non_static_tensor_shapes': len(bad_shapes),
    'arc_gen_holdout_policy': 'deterministic random seed 0, 60% of arc-gen; full arc-gen also reported',
    'validation': {
        'train': validate_split('train'),
        'test': validate_split('test'),
        'arc-gen_60pct_holdout': validate_examples(arcgen_holdout) if arcgen_holdout else None,
        'arc-gen_full': validate_split('arc-gen') if 'arc-gen' in task else None,
    },
}

print(json.dumps(summary, indent=2)[:8000])
with SUMMARY_PATH.open('w') as f:
    json.dump(summary, f, indent=2)

assert summary['validation']['train']['tensor_exact_zero_padded'][0] == summary['validation']['train']['tensor_exact_zero_padded'][1]
assert summary['validation']['test']['tensor_exact_zero_padded'][0] == summary['validation']['test']['tensor_exact_zero_padded'][1]
# task363 is intentionally train/test-consistent; its supplied arc-gen has a conflicting generator.
if TASK_ID != 'task363':
    assert summary['validation']['arc-gen_60pct_holdout']['tensor_exact_zero_padded'][0] == summary['validation']['arc-gen_60pct_holdout']['tensor_exact_zero_padded'][1]
    assert summary['validation']['arc-gen_full']['tensor_exact_zero_padded'][0] == summary['validation']['arc-gen_full']['tensor_exact_zero_padded'][1]

{
  "task_id": "task356",
  "rule": "generic connect all same-colored seed points that share a row or column",
  "onnx_path": "/kaggle/working/task356_revised_onnx/task356.onnx",
  "onnx_size_bytes": 86524,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "ops": {
    "Constant": 323,
    "ReduceSum": 19,
    "Greater": 46,
    "Cast": 64,
    "Slice": 45,
    "Mul": 65,
    "CumSum": 36,
    "GreaterOrEqual": 18,
    "Add": 27,
    "Clip": 19,
    "Sub": 1,
    "Concat": 1
  },
  "forbidden_ops": [],
  "empty_optional_inputs": 0,
  "non_static_tensor_shapes": 0,
  "arc_gen_holdout_policy": "deterministic random seed 0, 60% of arc-gen; full arc-gen also reported",
  "validation": {
    "train": {
      "tensor_exact_zero_padded": [
        3,
        3
      ],
      "grid_argmax_inside_canvas": [
        3,
        3
      ],
      "outside_input_active_all_channels_zero": [
        3,
        3
      ],
      "bad_indices"

In [12]:
with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')

print('Wrote:', SUBMISSION_PATH)
print('Zip contents:', zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f'{TASK_ID}.onnx']

Wrote: /kaggle/working/submission.zip
Zip contents: ['task356.onnx']
